# EEG-Based Neurodevelopmental Age Estimation and Analysis## Results Presentation NotebookThis notebook loads pre-saved results from the pipeline and presents them.It does **not** re-run any models — all data comes from CSV/JSON files in `results/`.**Dataset**: HBN-EEG (Healthy Brain Network, Child Mind Institute) — Releases R1–R11**Subjects**: 1,826 children and adolescents (ages 5–22)**EEG**: 129-channel resting-state recordings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os, json, sys
from IPython.display import display, Markdown, Image, HTML

# Locate directories
PROJECT_DIR = os.path.dirname(os.path.abspath('.'))
if 'neurodevelopment_project' not in PROJECT_DIR:
    PROJECT_DIR = os.path.join(PROJECT_DIR, 'neurodevelopment_project')
if not os.path.exists(os.path.join(PROJECT_DIR, 'results')):
    PROJECT_DIR = '.'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
REPO_ROOT = os.path.dirname(PROJECT_DIR) if os.path.basename(PROJECT_DIR) == 'neurodevelopment_project' else PROJECT_DIR
FEATURE_MATRIX = os.path.join(REPO_ROOT, 'data', 'processed', 'feature_matrix.csv')
print(f"Results: {RESULTS_DIR}")
print(f"Feature matrix: {FEATURE_MATRIX}")


---## 1. Dataset OverviewThe Healthy Brain Network (HBN) is a landmark biobank initiative by the **Child Mind Institute**collecting neuroimaging, behavioral, and cognitive data from children and adolescents in theNew York City area. This project uses the EEG component across all 11 data releases.

In [ ]:
# Load the raw feature matrix to show dataset statistics
fm = pd.read_csv(FEATURE_MATRIX)
print(f"Feature matrix dimensions: {fm.shape[0]:,} subjects × {fm.shape[1]:,} columns")


### 1.1 Subject Demographics

In [ ]:
# Demographics summary
demo = {
    'Metric': ['Total Subjects', 'Age Range', 'Mean Age ± SD', 'Median Age',
               'Male', 'Female', 'Sex Ratio (M:F)', 'Data Releases'],
    'Value': [
        f"{len(fm):,}",
        f"{fm['age'].min():.0f} – {fm['age'].max():.0f} years",
        f"{fm['age'].mean():.2f} ± {fm['age'].std():.2f} years",
        f"{fm['age'].median():.1f} years",
        f"{(fm['sex']=='M').sum():,} ({100*(fm['sex']=='M').mean():.1f}%)",
        f"{(fm['sex']=='F').sum():,} ({100*(fm['sex']=='F').mean():.1f}%)",
        f"{(fm['sex']=='M').sum()/(fm['sex']=='F').sum():.2f}",
        f"11 (R1–R11)",
    ]
}
display(pd.DataFrame(demo).style.hide(axis='index').set_properties(**{'text-align': 'left'}))


### 1.2 Age Distribution by Group

In [ ]:
# Age group breakdown
bins = [5, 8, 10, 12, 14, 17, 23]
labels = ['5-7', '8-9', '10-11', '12-13', '14-16', '17-22']
fm['age_group'] = pd.cut(fm['age'], bins=bins, labels=labels, right=False)

age_summary = fm.groupby('age_group', observed=False).agg(
    n=('age', 'count'),
    mean_age=('age', 'mean'),
    pct_male=('sex', lambda x: f"{100*(x=='M').mean():.1f}%"),
).reset_index()
age_summary['pct_of_total'] = (100 * age_summary['n'] / len(fm)).round(1).astype(str) + '%'
age_summary.columns = ['Age Group', 'n', 'Mean Age', '% Male', '% of Total']
display(age_summary.style.hide(axis='index').format({'Mean Age': '{:.1f}'}))


### 1.3 Subjects per Data Release

In [ ]:
rel_counts = fm['release'].value_counts().sort_index()
rel_df = pd.DataFrame({'Release': rel_counts.index, 'Subjects': rel_counts.values})
rel_df['% of Total'] = (100 * rel_df['Subjects'] / len(fm)).round(1)
display(rel_df.style.hide(axis='index'))


---## 2. EEG Feature Extraction & PreprocessingResting-state EEG (129 channels, EGI HydroCel) was processed into spectral and complexity features.

In [ ]:
# Feature breakdown
META_COLS = ['subject_id','attention','attention_high','p_factor',
             'internalizing','externalizing','age','sex','release',
             'attention_source_value','attention_source_scale']
BEHAV_COLS = ['symbolSearch_mean_rt','symbolSearch_std_rt','symbolSearch_accuracy',
              'symbolSearch_rt_iqr','seqLearning_mean_rt','seqLearning_std_rt',
              'seqLearning_accuracy','seqLearning_rt_iqr']

eeg_cols = [c for c in fm.columns if c not in META_COLS + BEHAV_COLS]

power = [c for c in eeg_cols if c.split('_')[0] in ['delta','theta','alpha','beta','gamma']
         and 'ratio' not in c and 'coh' not in c and 'entropy' not in c]
ratio = [c for c in eeg_cols if 'theta_alpha_ratio' in c]
spec_ent = [c for c in eeg_cols if 'spectral_entropy' in c]
samp_ent = [c for c in eeg_cols if 'sample_entropy' in c]
coh = [c for c in eeg_cols if 'alpha_coh' in c]

feat_table = pd.DataFrame({
    'Feature Family': ['Band Power (δ, θ, α, β, γ)', 'Theta/Alpha Ratio',
                        'Spectral Entropy', 'Sample Entropy', 'Alpha Coherence',
                        'TOTAL EEG Features'],
    'Count': [len(power), len(ratio), len(spec_ent), len(samp_ent), len(coh), len(eeg_cols)],
    'Channels': ['129 × 5 bands', '129 + 1 mean', '129', '129', '60 pairs', '—'],
    'Description': [
        'PSD via Welch method (V²/Hz)',
        'θ power / α power per channel',
        'Shannon entropy of power spectrum',
        'Regularity of raw time series',
        'Phase synchrony between channel pairs',
        '',
    ]
})
display(feat_table.style.hide(axis='index'))


### 2.1 Preprocessing Pipeline

In [ ]:
# Preprocessing stats
nan_in_coh = fm[coh].isna().mean()
dropped_coh = (nan_in_coh > 0.5).sum()
retained_coh = len(coh) - dropped_coh

total_missing = fm[eeg_cols].isna().sum().sum()
total_cells = len(fm) * len(eeg_cols)
retained_features = len(eeg_cols) - dropped_coh

preprocessing = pd.DataFrame({
    'Step': [
        '1. Load raw feature matrix',
        '2. Drop high-NaN features (>50% missing)',
        '3. Log₁₀ transform band-power features',
        '4. Compute seqLearning partial credit',
        '5. Median imputation (within CV folds)',
        '6. Standard scaling (Ridge only, within CV folds)',
    ],
    'Details': [
        f'{len(fm):,} subjects × {len(eeg_cols):,} EEG features',
        f'Dropped {dropped_coh} alpha coherence features (>98% NaN) → {retained_features:,} features retained',
        f'{len(power)} features rescaled from V²/Hz (~10⁻¹² scale) to log₁₀ scale',
        f'Position-wise scoring from {1129} raw event files → {660} subjects matched',
        f'Overall missing: {total_missing:,}/{total_cells:,} cells ({100*total_missing/total_cells:.2f}%)',
        f'Zero mean, unit variance — fitted on training fold only (no leakage)',
    ]
})
display(preprocessing.style.hide(axis='index'))


### 2.2 Target Variables Availability

In [ ]:
# Target availability
targets_info = []
for col, label in [('age', 'Chronological Age'), ('attention', 'Attention Problems'),
                    ('internalizing', 'Internalizing'), ('externalizing', 'Externalizing'),
                    ('p_factor', 'General Psychopathology (p-factor)'),
                    ('symbolSearch_accuracy', 'Symbol Search Accuracy'),
                    ('seqLearning_accuracy', 'Sequential Learning (exact match)')]:
    n = fm[col].notna().sum()
    targets_info.append({
        'Target': label,
        'Available': f'{n:,}',
        '% Available': f'{100*n/len(fm):.1f}%',
        'Mean ± SD': f'{fm[col].mean():.3f} ± {fm[col].std():.3f}' if n > 0 else '—',
    })

# Add partial credit (computed from raw files, not in feature matrix)
targets_info.append({
    'Target': 'Sequential Learning (partial credit)',
    'Available': '660',
    '% Available': '36.1%',
    'Mean ± SD': '0.669 ± 0.253',
})

display(pd.DataFrame(targets_info).style.hide(axis='index'))


### 2.3 Key Preprocessing Decisions| Decision | Rationale ||---|---|| **Log₁₀ transform** on band-power features | Raw PSD values are ~10⁻¹² V²/Hz with CV ≈ 0.7; log transform gives std ≈ 1.3–5.5, making them usable for ML || **Position-wise partial credit** for seqLearning | Exact-match scoring produces 76.2% zeros (too strict for 7–10 element sequences); partial credit gives mean = 0.67, continuous distribution || **Drop alpha coherence** features with >50% NaN | 10 of 60 coherence features have >98% missing; 50 retained || **Imputation inside CV folds** | Prevents data leakage — test fold statistics never influence training || **GroupKFold by release** as supplementary CV | Tests generalization across data collection batches (different sites/dates) |

---## 3. Pipeline Execution Summary

In [ ]:
summary_path = os.path.join(RESULTS_DIR, 'reports', 'pipeline_summary.md')
with open(summary_path, 'r') as f:
    display(Markdown(f.read()))


---## 4. Experiment 1: EEG-Based Age Prediction**Objective**: Predict chronological age from 1,083 EEG features using 10-fold CV.**Models**: Mean Baseline, Ridge Regression, Random Forest (300 trees).**Input**: 1,826 subjects × 1,083 EEG features.

### 4.1 Model Comparison

In [ ]:
metrics = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics', 'age_prediction_metrics.csv'))
display(metrics[['model', 'cv_type', 'R2', 'MAE', 'RMSE', 'Pearson_r']].round(4).style.hide(axis='index'))


### 4.2 Age Prediction Scatter PlotRandom Forest out-of-fold predictions vs actual age (10-fold CV, n=1,826).

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'age_prediction_scatter.png')))


### 4.3 Model Comparison Chart

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'model_comparison.png')))


### 4.4 Performance by Age Group

In [ ]:
age_grp = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'age_prediction_by_age_group.csv'))
display(age_grp[['age_group', 'n', 'R2', 'MAE', 'RMSE']].round(4).style.hide(axis='index'))


In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'age_prediction_by_age_group.png')))


### 4.5 Performance by Sex

In [ ]:
sex_df = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'age_prediction_by_sex.csv'))
display(sex_df[['sex', 'n', 'R2', 'MAE', 'RMSE']].round(4).style.hide(axis='index'))


### 4.6 Generalization Across Data Releases

In [ ]:
rel_df = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'age_prediction_by_release.csv'))
display(rel_df[['release', 'n', 'R2', 'MAE']].round(4).style.hide(axis='index'))


### 4.7 Interpretation- **Random Forest R² = 0.498, MAE = 1.82 years** — the model explains ~50% of age variance from EEG alone.- **GroupKFold R² = 0.494** — nearly identical, confirming the result is not a batch artifact.- Best performance for ages 8–11 (MAE ≈ 1.1 yr); worst for 17–22 (MAE = 4.6 yr, only 66 subjects).- Comparable performance across sexes (Male MAE = 1.84, Female MAE = 1.78).

---## 5. Experiment 2: Brain-Age Gap Analysis**Brain-age gap** = predicted age − actual age.Tests whether deviations from typical brain maturity associate with behavioral symptoms.

### 5.1 Brain-Age Gap Distribution

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'brain_age_gap_distribution.png')))


### 5.2 Age-Bias CorrectionBefore correction, gap correlates with age at r = −0.81 (regression to the mean).After residualization on age, r → 0.00.

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'gap_vs_age.png')))


### 5.3 Behavioral Associations (after age-bias correction)

In [ ]:
assoc = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'brain_age_associations.csv'))
display(assoc[['target', 'gap_type', 'r', 'p', 'ci_low', 'ci_high', 'p_bonferroni', 'n']].round(4).style.hide(axis='index'))


### 5.4 InterpretationAfter age-bias correction, the brain-age gap shows **no significant associations** with any behavioralmeasure (all |r| < 0.06, all p > 0.05 after Bonferroni correction for 4 tests). This indicates thatin this cross-sectional sample, EEG-derived brain maturity does not predict behavioral symptom severity.**Note**: A positive brain-age gap does NOT imply "accelerated aging" — it means the model's predictionwas higher than actual age, which could reflect noise, atypical but healthy EEG patterns, or other factors.

---## 6. Experiment 3: Feature ImportanceWhich EEG features drive the age prediction model?

### 6.1 Top 20 Individual FeaturesColor-coded by feature family. Both Gini (built-in RF) and permutation importance agree.

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'feature_importance_top20.png')))


### 6.2 Importance by Feature Family

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'importance_by_family.png')))


In [ ]:
perm_family = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'importance_by_family_permutation.csv'))
display(perm_family.round(5))


### 6.3 Feature-Age Correlations by Family

In [ ]:
corr_family = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'correlation_by_family.csv'))
display(corr_family.round(4))


### 6.4 Interpretation**Key insight**: The importance and correlation analyses tell complementary stories:- **Theta/alpha ratio** has the strongest *linear* age correlations (mean |r| = 0.28), consistent with the established theta-to-alpha shift during maturation.- **Delta power** features have the highest *model importance* (mean permutation importance 2.5×), because the Random Forest exploits nonlinear patterns across many delta channels jointly — patterns invisible to linear correlation.- **Sample entropy** ranks high on both measures, reflecting increasing neural complexity with age.

---## 7. Experiment 4: Cognitive Performance PredictionCan EEG predict cognitive task performance? Tested with 3 feature sets × 3 models.

### 7.1 Model Comparison

In [ ]:
cog_metrics = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics', 'cognitive_prediction_metrics.csv'))
display(cog_metrics[['target', 'feature_set', 'model', 'R2', 'MAE', 'Pearson_r', 'n']].round(4).style.hide(axis='index'))


In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'cognitive_prediction_comparison.png')))


### 7.2 Key Finding — Negative Result| Target | Best Model | R² | n ||---|---|---|---|| Symbol Search Accuracy | Age+Sex / Ridge | 0.081 | 899 || Sequential Learning (partial) | Age+Sex / Ridge | 0.052 | 660 || Any EEG model | — | < 0.045 | — |This is a **negative but scientifically informative result**:- Resting-state EEG features predict **age** well (R² = 0.50) but NOT **cognitive performance** (R² ≈ 0).- EEG adds **zero** predictive value beyond age and sex for either cognitive task.- Ridge with 1,083 EEG features goes deeply negative (R² = −1.5 to −2.5), confirming severe overfitting to noise.- The best cognitive predictors are simply age and sex (R² ≈ 0.05–0.08), reflecting known developmental improvements.

---## 8. Experiment 5: Developmental AnalysisHow do EEG features and cognitive scores vary across the age range?

### 8.1 EEG Feature Trends with Age

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'eeg_age_trends_by_family.png')))


In [ ]:
trends = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'eeg_age_trends_by_family.csv'))
display(trends.round(4))


### 8.2 Cognitive Performance Across Age Groups

In [ ]:
display(Image(os.path.join(RESULTS_DIR, 'figures', 'cognitive_age_curves.png')))


In [ ]:
cog_curves = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'cognitive_age_curves.csv'))
age_rows = cog_curves[~cog_curves['age_group'].str.startswith('all_')]
display(age_rows[['target', 'age_group', 'n', 'mean', 'std']].round(4).style.hide(axis='index'))


### 8.3 Brain-Age Gap and Cognitive Performance (Partial Correlations)

In [ ]:
gap_cog_path = os.path.join(RESULTS_DIR, 'tables', 'gap_cognitive_partial_correlations.csv')
if os.path.exists(gap_cog_path):
    gap_cog = pd.read_csv(gap_cog_path)
    display(gap_cog.round(4).style.hide(axis='index'))
else:
    print("Gap-cognitive results not available.")


### 8.4 Interpretation- **Theta/alpha ratio decreases with age** (r = −0.28): children's EEG shifts from slow (theta) to fast (alpha) rhythms — the most robust developmental EEG marker.- **Sample entropy and spectral entropy increase with age** (r ≈ +0.16): neural signals become more complex with maturation.- **Band power** shows minimal individual age trends (r ≈ 0) despite being collectively useful in the RF model — the model captures nonlinear multi-channel patterns.- **Brain-age gap does NOT predict cognition** after controlling for age (partial r < 0.04, p > 0.3).

---## 9. Summary of Findings### Dataset at a Glance| Metric | Value ||---|---|| Subjects | 1,826 (5–22 years, M=10.3, SD=3.3) || EEG Channels | 129 (HydroCel Geodesic Sensor Net) || EEG Features (after cleaning) | 1,083 || Feature Families | Band power (645), Theta/alpha ratio (130), Spectral entropy (129), Sample entropy (129), Alpha coherence (50) || Data Releases | 11 (R1–R11) || Missing EEG Data | 1.02% of cells |### Key Results| Experiment | Key Result ||---|---|| **1. Age Prediction** | RF R² = 0.498, MAE = 1.82 years; GroupKFold R² = 0.494 || **2. Brain-Age Gap** | No significant behavioral associations after age-bias correction (all p > 0.05) || **3. Feature Importance** | Delta power (RF importance) and theta/alpha ratio (age correlation) drive prediction || **4. Cognitive Prediction** | Near-zero R² — resting EEG does not predict cognitive performance beyond age || **5. Developmental Analysis** | Theta/alpha ratio tracks maturation (r = −0.28); gap unrelated to cognition |### Limitations1. **Cross-sectional design**: age-related differences reflect between-subject variation, not within-subject developmental trajectories2. **Resting-state EEG only**: task-related EEG may contain more cognitive-performance-relevant signal3. **Clinical-community sample** (HBN): results may not generalize to typically-developing populations4. **Feature extraction** was limited to PSD and basic complexity measures — connectivity, microstates, and deep learning on raw signals could improve results5. **Age imbalance**: 53% of subjects are aged 5–9; only 66 subjects (3.6%) are 17–22